# Aula 04 — Diagnósticos estatísticos e aderência à NBR 14653

Nesta aula, vamos ler a saída da Aula 3 e avaliar os diagnósticos do modelo.

## Objetivos

- consumir a saída da Aula 3;
- gerar o relatório de diagnósticos;
- analisar significância e resíduos;
- concluir a sequência do treinamento.

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Final

import pandas as pd

from servicos.carregamento import resolve_project_root
from servicos.regressao import fit_ols_regression
from servicos.diagnosticos import (
    build_nbr_diagnostics_report,
    format_coefficients_for_display,
    format_diagnostics_for_display,
)

In [ ]:
TARGET_COLUMN: Final[str] = 'preco'
PREFERRED_FEATURES: Final[tuple[str, ...]] = (
    'areaprivativa',
    'vagas',
    'distanciacentrokm',
    'dist_praia',
)


def locate_input_file(project_root: Path) -> Path:
    candidate = project_root / 'data' / 'output' / 'aula_03_amostra_com_residuos.csv'
    if candidate.exists():
        return candidate
    raise FileNotFoundError('Saída da Aula 3 não encontrada em data/output/.')

## Etapa 1 — Ler a saída da Aula 3

In [ ]:
project_root = resolve_project_root()
input_path = locate_input_file(project_root)
df_prepared = pd.read_csv(input_path)
print('Base carregada da Aula 3:', df_prepared.shape)
df_prepared.head()

## Etapa 2 — Ajustar e diagnosticar o modelo

In [ ]:
available_features = [column for column in PREFERRED_FEATURES if column in df_prepared.columns]
artifacts = fit_ols_regression(
    df=df_prepared,
    target_col=TARGET_COLUMN,
    feature_columns=available_features,
    add_intercept=True,
)

diagnostics_report, coefficients_report, summary = build_nbr_diagnostics_report(
    artifacts,
    minimum_adjusted_r_squared=0.70,
    alpha_f=0.05,
    alpha_t=0.10,
    alpha_shapiro=0.05,
    dw_lower=1.5,
    dw_upper=2.5,
)

## Etapa 3 — Ler os diagnósticos

In [ ]:
format_diagnostics_for_display(diagnostics_report)

In [ ]:
format_coefficients_for_display(coefficients_report)

## Etapa 4 — Resumo final

In [ ]:
summary

In [ ]:
print(f"Status geral do modelo: {summary['status_geral_modelo']}")
print(
    'Itens aprovados ou normais: '
    f"{summary['itens_aprovados_ou_normais']} de {summary['itens_avaliados']}"
)
print(f"Proporção de aprovação: {summary['proporcao_aprovacao']:.2%}")
print(
    'Coeficientes significativos a 10%: '
    f"{summary['coeficientes_significativos_10pct']} de {summary['coeficientes_totais']}"
)
print('Observação sobre normalidade:', summary['teste_normalidade_observacao'])
print('Faixa usada para Durbin-Watson:', summary['durbin_watson_faixa'])
print(f"Valor observado de Durbin-Watson: {summary['durbin_watson_valor']:.6f}")

## Conclusão

A Aula 4 fecha o ciclo do treinamento e consolida a leitura técnica do modelo.